# Correlations Analysis — Uber NCR Ride Bookings

**Questions from:** `process/active_questions.md` → Correlations

1. `Booking Value` and `Ride Distance` correlation is 0.01 — essentially zero. In a real ride-hailing dataset these should be strongly correlated. What explains this?
2. `Driver Ratings` and `Customer Rating` correlation is -0.00. Are these two independent random draws, or are they genuinely uncorrelated?

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../../..'))

import pandas as pd
import numpy as np
from pathlib import Path
from utils.analysis_log import AnalysisLog

OUTPUT_DIR = Path('../notebook_logs')
OUTPUT_DIR.mkdir(exist_ok=True)

log = AnalysisLog()
log.add(
    section_id='01_setup',
    title='Setup',
    cell_type='setup',
    purpose='Load libraries and initialize AnalysisLog for correlations investigation',
    data={'libraries': ['pandas', 'numpy']},
)

## 1. Load Data (Completed rides only)

In [ ]:
DATA_PATH = Path('..') / 'data' / 'ncr_ride_bookings.csv'
df = pd.read_csv(DATA_PATH)
completed = df[df['Booking Status'] == 'Completed'].copy()

log.add(
    section_id='02_load_data',
    title='Load Data',
    cell_type='setup',
    purpose='Load dataset and filter to completed rides where both Booking Value and Ride Distance are non-null',
    data={'total_rows': len(df), 'completed_rows': len(completed)},
)
completed.shape

## 2. Booking Value vs Ride Distance — Top-Level Correlation

Reproduce the near-zero correlation, then test whether non-linear relationships exist.

In [ ]:
pearson_r = completed['Booking Value'].corr(completed['Ride Distance'])
# Spearman = Pearson on ranks (no scipy needed)
spearman_r = completed['Booking Value'].rank().corr(completed['Ride Distance'].rank())

corr_result = {
    'pearson': round(pearson_r, 4),
    'spearman_via_rank': round(spearman_r, 4),
    'n': len(completed),
}

log.add(
    section_id='03_fare_distance_corr',
    title='Booking Value vs Ride Distance — Pearson and Spearman',
    cell_type='analysis',
    purpose='Test linear and rank-based correlations between fare and distance to rule out non-linear relationships masking a real signal',
    data=corr_result,
)
corr_result

## 3. Fare vs Distance — Distribution Check

Are these independently drawn from separate uniform or random distributions? Real fare is a function of distance + time + surge — it should not be independently generated.

In [ ]:
fare_stats = completed['Booking Value'].describe().round(2).to_dict()
dist_stats = completed['Ride Distance'].describe().round(4).to_dict()

# Check if distributions look uniform: for a uniform(a,b), mean ≈ (a+b)/2, std ≈ (b-a)/sqrt(12)
fare_min, fare_max = completed['Booking Value'].min(), completed['Booking Value'].max()
dist_min, dist_max = completed['Ride Distance'].min(), completed['Ride Distance'].max()

uniform_fare_expected_mean = (fare_min + fare_max) / 2
uniform_fare_expected_std = (fare_max - fare_min) / (12 ** 0.5)
uniform_dist_expected_mean = (dist_min + dist_max) / 2
uniform_dist_expected_std = (dist_max - dist_min) / (12 ** 0.5)

dist_check = {
    'booking_value': {
        'actual_mean': round(float(completed['Booking Value'].mean()), 2),
        'actual_std': round(float(completed['Booking Value'].std()), 2),
        'uniform_expected_mean': round(uniform_fare_expected_mean, 2),
        'uniform_expected_std': round(uniform_fare_expected_std, 2),
    },
    'ride_distance': {
        'actual_mean': round(float(completed['Ride Distance'].mean()), 4),
        'actual_std': round(float(completed['Ride Distance'].std()), 4),
        'uniform_expected_mean': round(uniform_dist_expected_mean, 4),
        'uniform_expected_std': round(uniform_dist_expected_std, 4),
    },
}

log.add(
    section_id='04_distribution_check',
    title='Fare and Distance — Uniform Distribution Test',
    cell_type='analysis',
    purpose='Compare actual mean/std against what a uniform distribution over the same range would produce, to detect independently generated columns',
    data=dist_check,
)
dist_check

## 4. Fare vs Distance — By Vehicle Type

If different vehicle types have different per-km rates, pooling them all could suppress correlation. Check correlation within each vehicle type.

In [ ]:
corr_by_vehicle = (
    completed.groupby('Vehicle Type')
    .apply(lambda g: pd.Series({
        'n': len(g),
        'pearson': round(g['Booking Value'].corr(g['Ride Distance']), 4),
        'spearman': round(g['Booking Value'].rank().corr(g['Ride Distance'].rank()), 4),
        'fare_mean': round(g['Booking Value'].mean(), 2),
        'dist_mean': round(g['Ride Distance'].mean(), 2),
    }))
    .reset_index()
)

log.add(
    section_id='05_corr_by_vehicle',
    title='Fare vs Distance Correlation by Vehicle Type',
    cell_type='analysis',
    purpose='Test whether pooling vehicle types with different per-km rates suppresses the fare-distance correlation',
    data=corr_by_vehicle.to_dict(orient='records'),
)
corr_by_vehicle

## 5. Fare vs Distance — Decile Analysis

Split Ride Distance into deciles. If distance drives fare, mean fare should rise monotonically across deciles.

In [ ]:
completed['dist_decile'] = pd.qcut(completed['Ride Distance'], q=10, labels=False)
decile_stats = (
    completed.groupby('dist_decile')
    .agg(
        n=('Ride Distance', 'count'),
        dist_min=('Ride Distance', 'min'),
        dist_max=('Ride Distance', 'max'),
        fare_mean=('Booking Value', 'mean'),
        fare_median=('Booking Value', 'median'),
    )
    .round(2)
)

log.add(
    section_id='06_decile_analysis',
    title='Mean Fare by Distance Decile',
    cell_type='analysis',
    purpose='Check whether mean fare rises monotonically across distance deciles, which would confirm a real relationship masked by noise in the raw correlation',
    data=decile_stats.to_dict(orient='index'),
)
decile_stats

## 6. Fare vs Distance — Implied Per-km Rate

If fare = base_fare + rate × distance, the per-km rate (fare / distance) should cluster around a consistent value per vehicle type. If instead the per-km rate has enormous variance, fare is being generated independently of distance.

In [ ]:
completed['implied_rate'] = completed['Booking Value'] / completed['Ride Distance']

rate_by_vehicle = (
    completed.groupby('Vehicle Type')['implied_rate']
    .agg(['mean', 'std', 'min', 'max',
          lambda x: x.quantile(0.25),
          lambda x: x.quantile(0.75)])
    .round(2)
)
rate_by_vehicle.columns = ['mean', 'std', 'min', 'max', 'p25', 'p75']

overall_rate = {
    'mean': round(float(completed['implied_rate'].mean()), 2),
    'std': round(float(completed['implied_rate'].std()), 2),
    'cv': round(float(completed['implied_rate'].std() / completed['implied_rate'].mean()), 3),
    'min': round(float(completed['implied_rate'].min()), 2),
    'max': round(float(completed['implied_rate'].max()), 2),
}

log.add(
    section_id='07_implied_rate',
    title='Implied Per-km Rate (Fare / Distance)',
    cell_type='analysis',
    purpose='Check whether fare/distance clusters around a consistent per-km rate per vehicle type; extreme variance indicates fare and distance are independently generated',
    data={
        'overall': overall_rate,
        'by_vehicle': rate_by_vehicle.to_dict(orient='index'),
    },
)
rate_by_vehicle

## 7. Driver Rating vs Customer Rating — Top-Level Correlation

In [ ]:
rating_pearson = completed['Driver Ratings'].corr(completed['Customer Rating'])
rating_spearman = completed['Driver Ratings'].rank().corr(completed['Customer Rating'].rank())

rating_corr = {
    'pearson': round(rating_pearson, 4),
    'spearman_via_rank': round(rating_spearman, 4),
    'n': int(completed[['Driver Ratings', 'Customer Rating']].dropna().shape[0]),
}

log.add(
    section_id='08_rating_corr',
    title='Driver Rating vs Customer Rating — Pearson and Spearman',
    cell_type='analysis',
    purpose='Confirm whether the near-zero correlation between driver and customer ratings holds under both linear and rank-based methods',
    data=rating_corr,
)
rating_corr

## 8. Rating Joint Distribution — Value Pair Frequency

If both ratings are independently drawn, each (driver_rating, customer_rating) pair should appear at roughly equal frequency. If they co-vary, certain diagonals (matching values) should dominate.

In [ ]:
rating_pairs = (
    completed.groupby(['Driver Ratings', 'Customer Rating'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

n_possible_pairs = completed['Driver Ratings'].nunique() * completed['Customer Rating'].nunique()
n_observed_pairs = len(rating_pairs)

# Count pairs where both ratings are identical
same_value_pairs = rating_pairs[rating_pairs['Driver Ratings'] == rating_pairs['Customer Rating']]
same_value_count = same_value_pairs['count'].sum()
total_ratings = rating_pairs['count'].sum()

pair_summary = {
    'possible_pairs': int(n_possible_pairs),
    'observed_pairs': int(n_observed_pairs),
    'top_10_pairs': rating_pairs.head(10).to_dict(orient='records'),
    'rows_where_driver_rating_equals_customer_rating': int(same_value_count),
    'pct_matching_ratings': round(same_value_count / total_ratings * 100, 1),
    'expected_pct_if_independent': round(100 / completed['Driver Ratings'].nunique(), 1),
}

log.add(
    section_id='09_rating_pair_distribution',
    title='Driver vs Customer Rating — Joint Value Pair Frequency',
    cell_type='analysis',
    purpose='Check the joint distribution of rating pairs to determine if ratings are independently generated or if matching/diagonal pairs are over-represented',
    data=pair_summary,
)
pair_summary

## 9. Rating Marginal Distributions — Unique Values and Frequency

Real ratings collect at round numbers (4.0, 4.5, 5.0). If each value appears with equal frequency, they were drawn uniformly.

In [ ]:
driver_rating_dist = completed['Driver Ratings'].value_counts().sort_index()
customer_rating_dist = completed['Customer Rating'].value_counts().sort_index()

driver_cv = driver_rating_dist.std() / driver_rating_dist.mean()
customer_cv = customer_rating_dist.std() / customer_rating_dist.mean()

rating_dist_result = {
    'driver_rating': {
        'nunique': int(completed['Driver Ratings'].nunique()),
        'value_counts': driver_rating_dist.to_dict(),
        'cv_of_frequencies': round(float(driver_cv), 4),
    },
    'customer_rating': {
        'nunique': int(completed['Customer Rating'].nunique()),
        'value_counts': customer_rating_dist.to_dict(),
        'cv_of_frequencies': round(float(customer_cv), 4),
    },
}

log.add(
    section_id='10_rating_marginal_dist',
    title='Rating Marginal Distributions',
    cell_type='analysis',
    purpose='Examine the frequency of each rating value for drivers and customers to detect uniform generation vs realistic skewed distribution',
    data=rating_dist_result,
)
rating_dist_result

## 10. Rating — Conditional Correlation by Rating Tier

Split rides into low/mid/high rating tiers for the driver. If customer ratings track driver tiers, there is a suppressed relationship.

In [ ]:
completed['driver_tier'] = pd.cut(
    completed['Driver Ratings'],
    bins=[2.9, 3.5, 4.0, 4.5, 5.01],
    labels=['3.0–3.5', '3.5–4.0', '4.0–4.5', '4.5–5.0'],
)

tier_stats = (
    completed.groupby('driver_tier', observed=True)
    .agg(
        n=('Customer Rating', 'count'),
        customer_mean=('Customer Rating', 'mean'),
        customer_std=('Customer Rating', 'std'),
        driver_mean=('Driver Ratings', 'mean'),
    )
    .round(3)
)

log.add(
    section_id='11_conditional_corr_by_tier',
    title='Customer Rating Mean by Driver Rating Tier',
    cell_type='analysis',
    purpose='Check whether customer ratings systematically differ across driver rating tiers, indicating a suppressed relationship not visible in the raw correlation',
    data=tier_stats.to_dict(orient='index'),
)
tier_stats

## 11. Rating — Are Values Drawn from a Fixed Discrete Set?

Check whether ratings are restricted to multiples of 0.1 (as `nunique=21` from 3.0–5.0 suggests) and whether the step size is perfectly uniform.

In [ ]:
driver_vals = sorted(completed['Driver Ratings'].dropna().unique())
customer_vals = sorted(completed['Customer Rating'].dropna().unique())

driver_steps = [round(driver_vals[i+1] - driver_vals[i], 4) for i in range(len(driver_vals)-1)]
customer_steps = [round(customer_vals[i+1] - customer_vals[i], 4) for i in range(len(customer_vals)-1)]

discrete_check = {
    'driver_rating_unique_values': driver_vals,
    'driver_step_sizes': list(set(driver_steps)),
    'customer_rating_unique_values': customer_vals,
    'customer_step_sizes': list(set(customer_steps)),
    'driver_range': [driver_vals[0], driver_vals[-1]],
    'customer_range': [customer_vals[0], customer_vals[-1]],
}

log.add(
    section_id='12_discrete_value_check',
    title='Rating Discrete Value Structure',
    cell_type='analysis',
    purpose='Verify that both rating columns use a fixed discrete scale with uniform step size, and confirm the exact values present',
    data=discrete_check,
)
discrete_check

---
## Save Output

In [ ]:
log.save(OUTPUT_DIR / 'correlations_analysis.json')